In [ ]:
# Run from the repository root so that `utils` and the relative data paths resolve
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

from glob import glob
import pickle
from pathlib import Path

In [ ]:
test_folder = glob("dataset/test_folder_daytime/*.pkl")
test_folder.sort()

In [ ]:
my_detection_list = []
gt_list = []

for fname in test_folder:
    fname = Path(fname)
    with open(fname, 'rb') as f:  # Open file in binary read mode
        pk = pickle.load(f)
        test_signal = pk["signal"]  # Deserialize the signal data
        test_label = pk["label"]
    
    irmos_detector = np.load(f"runs/peak_count/Test/{fname.stem}_norm.npy")
    irmos_detector_result = np.sum(irmos_detector, axis=0).astype(np.int32)
    # VDC23-Lane2-Left; VDC21-Lane1-Right;
    strain_left = test_signal[1,:,1] # Lane2
    strain_right = test_signal[1,:,6] # Lane1

    

    peaks_car_left, _ = find_peaks(strain_left, height=(0.04,0.4), distance=10)
    peaks_car_right, _ = find_peaks(strain_right, height=(0.035,0.1), distance=10)
    peaks_truck_left, _ = find_peaks(strain_left, height=(0.4,1.0), distance=10)
    peaks_truck_right, _ = find_peaks(strain_right, height=(0.1,1.0), distance=10)

    my_detection = [len(peaks_car_left), len(peaks_car_right), len(peaks_truck_left), len(peaks_truck_right)]

    my_detection_list.append(np.array(my_detection))
    gt_list.append(np.sum(test_label,axis=0).astype(np.int32))
    
    print(f"\nMy detection:    {np.array(my_detection)}")
    print("Irmos count: ", irmos_detector_result)
    print("Ground truth: ", np.sum(test_label,axis=0).astype(np.int32))

In [ ]:
def eval_peak_count(granuity="2cls"):
    my_detection = np.array(my_detection_list).T
    gt = np.array(gt_list).T

    if granuity == "2cls":
        pred_light = np.sum(my_detection[[0,1],:], axis=0, keepdims=True)
        gt_light = np.sum(gt[[0,1],:], axis=0, keepdims=True)

        pred_heavy = my_detection[[3],:]
        gt_heavy = gt[[3],:]

        pred_to_compute = np.concatenate([pred_light, pred_heavy], axis=0)  # (2, 24)
        gt_to_compute = np.concatenate([gt_light, gt_heavy], axis=0)  # (2, 24)
    elif granuity == "4cls":
        pred_to_compute = my_detection  # (4, 24)
        gt_to_compute = gt  # (4, 24)

    mae = np.mean(np.abs(pred_to_compute - gt_to_compute), axis=1)


    # for b in [1, 5,10,20,30,60,600]:
    max_values = np.maximum(pred_to_compute, gt_to_compute)
    min_values = np.minimum(pred_to_compute, gt_to_compute)

    sum_max = np.sum(max_values, axis=1)
    sum_min = np.sum(min_values, axis=1)
    accuracy = 100*sum_min/sum_max


    # Compute mean gt
    mean_gt = np.mean(gt_to_compute, axis=1)


    rmae = 100* mae / mean_gt

    print(f"MAE: {mae.round(2)}, Relative MAE: {rmae.round(2)}, Accuracy: {accuracy.round(2)}")

eval_peak_count(granuity="2cls")
eval_peak_count(granuity="4cls")